## Обучение XLM-RoBERTa base с балансировкой классов и ранней остановкой
Скрипт дообучает модель `FacebookAI/xlm-roberta-base` для трёхклассовой классификации тональности на данных `train_super/val_super`, используя символьные тексты, class weights для учёта дисбаланса, градиентное накопление и линейный scheduler с warmup. Качество отслеживается по macro-F1 на валидации, применяется early stopping, а лучшая модель и история обучения сохраняются в каталог `xlm_roberta_base_manual`.


In [ ]:
import os
import time
import random
import math
from typing import List, Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

TRAIN_CLEAN_PATH = "../data/processed/train_super.csv"
VAL_CLEAN_PATH   = "../data/processed/val_super.csv"

MODEL_ID   = "FacebookAI/xlm-roberta-base"
OUTPUT_DIR = "../models/xlm_roberta_base_manual"

NUM_LABELS         = 3
MAX_LEN            = 160      

EPOCHS             = 5        
PATIENCE           = 2       

BATCH_SIZE         = 8       
GRAD_ACCUM_STEPS   = 4       

LEARNING_RATE      = 2e-5    
WARMUP_RATIO       = 0.06
WEIGHT_DECAY       = 0.01     

USE_CLASS_WEIGHTS  = True    
LABEL_SMOOTHING    = 0.0     

RANDOM_STATE       = 42
MAX_TRAIN_SAMPLES  = 30000     


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class TextDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    device,
    grad_accum_steps: int = 1,
    epoch_idx: int = 1,
    class_weights: torch.Tensor | None = None,
    label_smoothing: float = 0.0,
):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    optimizer.zero_grad()

    num_batches = len(dataloader)
    start_time = time.time()

    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        logits = outputs.logits

        loss = F.cross_entropy(
            logits,
            batch["labels"],
            weight=class_weights,
            label_smoothing=label_smoothing,
        )

        loss = loss / grad_accum_steps
        loss.backward()

        total_loss += loss.item() * grad_accum_steps

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if device.type == "mps":
                torch.mps.empty_cache()

        preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
        labels = batch["labels"].detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels)

        if (step + 1) % 200 == 0 or (step + 1) == num_batches:
            elapsed = (time.time() - start_time) / 60
            avg_loss_so_far = total_loss / (step + 1)
            progress = 100.0 * (step + 1) / num_batches
            lr = scheduler.get_last_lr()[0]
            print(
                f"[Train] Epoch {epoch_idx} | "
                f"batch {step+1}/{num_batches} ({progress:.1f}%) | "
                f"avg_loss: {avg_loss_so_far:.4f} | lr: {lr:.6f} | "
                f"elapsed: {elapsed:.1f} мин"
            )

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def eval_one_epoch(
    model,
    dataloader,
    device,
    epoch_idx: int = 1,
    class_weights: torch.Tensor | None = None,
    label_smoothing: float = 0.0,
):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    num_batches = len(dataloader)
    start_time = time.time()

    with torch.no_grad():
        for step, batch in enumerate(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            logits = outputs.logits

            loss = F.cross_entropy(
                logits,
                batch["labels"],
                weight=class_weights,
                label_smoothing=label_smoothing,
            )
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
            labels = batch["labels"].detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels)

            if (step + 1) % 200 == 0 or (step + 1) == num_batches:
                elapsed = (time.time() - start_time) / 60
                avg_loss_so_far = total_loss / (step + 1)
                progress = 100.0 * (step + 1) / num_batches
                print(
                    f"[Val] Epoch {epoch_idx} | "
                    f"batch {step+1}/{num_batches} ({progress:.1f}%) | "
                    f"avg_loss: {avg_loss_so_far:.4f} | "
                    f"elapsed: {elapsed:.1f} мин"
                )

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def main():
    set_seed(RANDOM_STATE)

    assert os.path.exists(TRAIN_CLEAN_PATH), f"Нет файла {TRAIN_CLEAN_PATH}"
    assert os.path.exists(VAL_CLEAN_PATH),   f"Нет файла {VAL_CLEAN_PATH}"

    print("Читаем данные...")
    train_df = pd.read_csv(TRAIN_CLEAN_PATH)
    val_df   = pd.read_csv(VAL_CLEAN_PATH)

    for col in ["text", "label"]:
        if col not in train_df.columns:
            raise ValueError(f"В train_super нет колонки '{col}'")
        if col not in val_df.columns:
            raise ValueError(f"В val_super нет колонки '{col}'")

    if MAX_TRAIN_SAMPLES is not None and len(train_df) > MAX_TRAIN_SAMPLES:
        print(f"train_super имеет {len(train_df)} строк, "
              f"режем до {MAX_TRAIN_SAMPLES} для обучения...")
        train_df = train_df.sample(
            n=MAX_TRAIN_SAMPLES,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)

    print("Размер train:", train_df.shape)
    print("Размер val:  ", val_df.shape)
    print("\nРаспределение классов (train):")
    print(train_df["label"].value_counts().sort_index())
    print("\nРаспределение классов (val):")
    print(val_df["label"].value_counts().sort_index())

    # устройство
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("\nИспользуем устройство:", device)

    print("\nЗагружаем токенизатор и модель:", MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
    )
    model.to(device)

    train_dataset = TextDataset(
        texts=train_df["text"].tolist(),
        labels=train_df["label"].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )
    val_dataset = TextDataset(
        texts=val_df["text"].tolist(),
        labels=val_df["label"].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    # class weights
    if USE_CLASS_WEIGHTS:
        label_counts = train_df["label"].value_counts().sort_index()
        total = label_counts.sum()
        weights = [total / c for c in label_counts.tolist()]
        class_weights = torch.tensor(weights, dtype=torch.float, device=device)
        print("\nClass weights:", weights)
    else:
        class_weights = None
        print("\nClass weights: не используются")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_optim_steps = num_update_steps_per_epoch * EPOCHS
    num_warmup_steps = int(WARMUP_RATIO * total_optim_steps)

    print(f"\nВсего optimizer steps: {total_optim_steps}, "
          f"warmup steps: {num_warmup_steps}")

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_optim_steps,
    )

    best_f1 = -1.0
    history = []
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        print(f"\n===== Эпоха {epoch}/{EPOCHS} =====")
        start_time = time.time()

        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            epoch_idx=epoch,
            class_weights=class_weights,
            label_smoothing=LABEL_SMOOTHING,
        )
        val_loss, val_acc, val_f1 = eval_one_epoch(
            model,
            val_loader,
            device,
            epoch_idx=epoch,
            class_weights=class_weights,
            label_smoothing=LABEL_SMOOTHING,
        )

        elapsed = time.time() - start_time
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"\nTrain | loss: {train_loss:.4f} | acc: {train_acc:.4f} | f1_macro: {train_f1:.4f}"
        )
        print(
            f"Val   | loss: {val_loss:.4f} | acc: {val_acc:.4f} | f1_macro: {val_f1:.4f}"
        )
        print(f"LR: {current_lr:.6f}")
        print(f"Время эпохи: {elapsed/60:.2f} мин")

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "train_f1": train_f1,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "val_f1": val_f1,
                "lr": current_lr,
                "time_min": elapsed / 60,
            }
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            no_improve = 0
            print(f"🔥 Новый лучший F1: {best_f1:.4f}, сохраняем модель...")
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
        else:
            no_improve += 1
            print(
                f"val_f1 не улучшился ({val_f1:.4f} <= {best_f1:.4f}), "
                f"no_improve = {no_improve}"
            )
            if no_improve >= PATIENCE:
                print("⏹ Early stopping по F1")
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(
        os.path.join(OUTPUT_DIR, "training_history_xlm_roberta_base.csv"),
        index=False
    )

    print("\nОбучение завершено.")
    print(f"Лучшая macro-F1 на валидации: {best_f1:.4f}")
    print("Модель сохранена в:", OUTPUT_DIR)
    print("История обучения сохранена в:",
          os.path.join(OUTPUT_DIR, "training_history_xlm_roberta_base.csv"))


if __name__ == "__main__":
    main()


/opt/anaconda3/envs/sentiment/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Читаем данные...
train_super имеет 171111 строк, режем до 30000 для обучения...
Размер train: (30000, 4)
Размер val:   (18898, 4)

Распределение классов (train):
label
0     9782
1    10108
2    10110
Name: count, dtype: int64

Распределение классов (val):
label
0    6259
1    6302
2    6337
Name: count, dtype: int64

Используем устройство: mps

Загружаем токенизатор и модель: FacebookAI/xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Class weights: [np.float64(3.0668574933551422), np.float64(2.96794618124258), np.float64(2.9673590504451037)]

Всего optimizer steps: 4690, warmup steps: 281

===== Эпоха 1/5 =====
[Train] Epoch 1 | batch 200/3750 (5.3%) | avg_loss: 1.1078 | lr: 0.000004 | elapsed: 1.1 мин
[Train] Epoch 1 | batch 400/3750 (10.7%) | avg_loss: 1.0953 | lr: 0.000007 | elapsed: 2.1 мин
[Train] Epoch 1 | batch 600/3750 (16.0%) | avg_loss: 1.0507 | lr: 0.000011 | elapsed: 3.2 мин
[Train] Epoch 1 | batch 800/3750 (21.3%) | avg_loss: 1.0265 | lr: 0.000014 | elapsed: 4.4 мин
[Train] Epoch 1 | batch 1000/3750 (26.7%) | avg_loss: 0.9968 | lr: 0.000018 | elapsed: 5.5 мин
[Train] Epoch 1 | batch 1200/3750 (32.0%) | avg_loss: 0.9733 | lr: 0.000020 | elapsed: 6.6 мин
[Train] Epoch 1 | batch 1400/3750 (37.3%) | avg_loss: 0.9492 | lr: 0.000020 | elapsed: 7.7 мин
[Train] Epoch 1 | batch 1600/3750 (42.7%) | avg_loss: 0.9272 | lr: 0.000019 | elapsed: 8.7 мин
[Train] Epoch 1 | batch 1800/3750 (48.0%) | avg_loss: 0.9054 | 

KeyboardInterrupt: 

In [3]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader


# ===== НАСТРОЙКИ =====
VAL_PATH = "../data/processed/val_super.csv"
MODEL_DIR = "../models/xlm_roberta_base_manual"   # <-- твоя сохранённая модель
MAX_LEN = 192
BATCH_SIZE = 16
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")


# ===== ДАТАСЕТ =====
class InferenceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ===== ИНФЕРЕНС =====
def evaluate_model():
    assert os.path.exists(VAL_PATH), f"Не найден {VAL_PATH}"
    assert os.path.exists(MODEL_DIR), f"Нет модели в {MODEL_DIR}"

    print(" Загружаем валидацию...")
    df = pd.read_csv(VAL_PATH)
    texts = df["text"].astype(str).tolist()
    labels = df["label"].tolist()

    print(" Загружаем модель:", MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(DEVICE)
    model.eval()

    ds = InferenceDataset(texts, labels, tokenizer, MAX_LEN)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

    all_preds = []
    all_labels = []

    print("\n Делаем предсказания на валидации...")
    with torch.no_grad():
        for batch in dl:
            input_ids = batch["input_ids"].to(DEVICE)
            att = batch["attention_mask"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=att)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    # ===== МЕТРИКИ =====
    f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)

    print("\n===  Результаты на валидации ===")
    print("Accuracy:", round(acc, 4))
    print("Macro-F1:", round(f1, 4))
    print("\nClassification report:")
    print(classification_report(all_labels, all_preds, digits=4))

    return f1


if __name__ == "__main__":
    evaluate_model()


📄 Загружаем валидацию...
 Загружаем модель: ../models/xlm_roberta_base_manual


The tokenizer you are loading from '../models/xlm_roberta_base_manual' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



 Делаем предсказания на валидации...

===  Результаты на валидации ===
Accuracy: 0.7225
Macro-F1: 0.7213

Classification report:
              precision    recall  f1-score   support

           0     0.6164    0.6191    0.6178      6259
           1     0.7880    0.8466    0.8162      6302
           2     0.7607    0.7013    0.7298      6337

    accuracy                         0.7225     18898
   macro avg     0.7217    0.7223    0.7213     18898
weighted avg     0.7220    0.7225    0.7215     18898

